# Unidade 2 - Bloco prático da Aula 02: Grid vs Random sob nested CV

Compara Grid Search (27 combinações) e Random Search (10 sorteios) numa floresta aleatória, com nested cross-validation como juiz. Repare na diferença entre o score interno da busca e o score nested: é o selection bias medido ao vivo.

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_score)
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint

X, y = load_breast_cancer(return_X_y=True)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grade = {"n_estimators": [50, 100, 200],
         "max_depth": [3, 6, None],
         "min_samples_leaf": [1, 3, 10]}
dist = {"n_estimators": randint(50, 250),
        "max_depth": [3, 4, 5, 6, 8, 10, None],
        "min_samples_leaf": randint(1, 12)}

rf = RandomForestClassifier(random_state=42)
gs = GridSearchCV(rf, grade, cv=inner, scoring="f1", n_jobs=-1)
rs = RandomizedSearchCV(rf, dist, n_iter=10, cv=inner, scoring="f1",
                        n_jobs=-1, random_state=42)

t0 = time.time(); gs.fit(X, y); t_gs = time.time() - t0
t0 = time.time(); rs.fit(X, y); t_rs = time.time() - t0
print(f"Grid   (27 combos): melhor F1 interno = {gs.best_score_:.3f} "
      f"| tempo {t_gs:.0f}s")
print(f"Random (10 combos): melhor F1 interno = {rs.best_score_:.3f} "
      f"| tempo {t_rs:.0f}s")

f1_gs = cross_val_score(GridSearchCV(rf, grade, cv=inner, scoring="f1",
                                     n_jobs=-1),
                        X, y, cv=outer, scoring="f1").mean()
f1_rs = cross_val_score(RandomizedSearchCV(rf, dist, n_iter=10, cv=inner,
                                           scoring="f1", n_jobs=-1,
                                           random_state=42),
                        X, y, cv=outer, scoring="f1").mean()
print(f"Nested CV Grid:   F1 = {f1_gs:.3f}")
print(f"Nested CV Random: F1 = {f1_rs:.3f}")